In [ ]:
# Cell 1 — Setup
"""
04_evaluation.ipynb
===================
Interactive evaluation: stratified metrics, uncertainty, and applicability domain.

Scripted equivalents:
- `scripts/evaluate_complete.py`
- `scripts/error_analysis.py`
- `scripts/generate_paper_figures.py`
- `scripts/generate_supplementary.py`
- `scripts/validate_physics.py`
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.inference import load_model, predict_solubility
from tgnn_solv.evaluate import Evaluator
from tgnn_solv.uncertainty import MCDropoutPredictor, calibration_report
from tgnn_solv.domain import ApplicabilityDomain
from tgnn_solv.data import PROCESSED_DIR, make_loaders

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


In [ ]:
# Cell 2 — Load model and data
# Default evaluation target: scaffold split.
# For fair baseline comparison, point these reads to the solute split instead.
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, cfg = load_model(str(MODEL_PATH), DEVICE)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, batch_size=cfg.batch_size,
)

print(f"Model: {MODEL_PATH}")
print(f"Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}")


In [ ]:
# Cell 3 — Stratified evaluation
evaluator = Evaluator(model, cfg)
report = evaluator.evaluate(test_loader, test_df)
evaluator.print_report(report)

In [ ]:
# Cell 4 — Parity plot with error density

# Collect all predictions
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for sol_b, slv_b, tgt in test_loader:
        sol_b = sol_b.to(DEVICE)
        slv_b = slv_b.to(DEVICE)
        T = tgt["T"].to(DEVICE)
        mask = tgt["has_solubility"].to(DEVICE)
        mask_cpu = mask.detach().cpu()
        
        out = model(sol_b, slv_b, T)
        if mask.any():
            all_pred.append(out["ln_x2"][mask].detach().cpu().numpy())
            all_true.append(tgt["ln_x2"][mask_cpu].cpu().numpy())

pred = np.concatenate(all_pred)
true = np.concatenate(all_true)
errors = pred - true

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Parity plot
ax = axes[0]
ax.scatter(true, pred, s=3, alpha=0.3, c="steelblue")
lims = [min(true.min(), pred.min()) - 1, max(true.max(), pred.max()) + 1]
ax.plot(lims, lims, "r--", lw=1, label="y = x")
ax.plot(lims, [l + 1 for l in lims], "r:", lw=0.5, alpha=0.5)
ax.plot(lims, [l - 1 for l in lims], "r:", lw=0.5, alpha=0.5)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Experimental ln(x₂)")
ax.set_ylabel("Predicted ln(x₂)")
mae = np.abs(errors).mean()
r2 = 1 - (errors ** 2).sum() / ((true - true.mean()) ** 2).sum()
ax.set_title(f"Parity (MAE={mae:.3f}, R²={r2:.3f})")
ax.legend()
ax.set_aspect("equal")

# Error distribution
ax = axes[1]
ax.hist(errors, bins=80, color="coral", edgecolor="white", density=True)
ax.axvline(0, color="black", lw=1)
ax.axvline(errors.mean(), color="red", ls="--",
           label=f"bias={errors.mean():.3f}")
ax.set_xlabel("Error (pred − true)")
ax.set_ylabel("Density")
ax.set_title("Error distribution")
ax.legend()

# Error vs true value
ax = axes[2]
ax.scatter(true, np.abs(errors), s=3, alpha=0.3, c="seagreen")
ax.set_xlabel("Experimental ln(x₂)")
ax.set_ylabel("|Error|")
ax.set_title("Error magnitude vs solubility")
ax.axhline(1.0, color="red", ls=":", label="|error| = 1")
ax.legend()

plt.tight_layout()
plt.savefig(str(NOTEBOOK_FIG_DIR / "evaluation_plots.png"), dpi=150)
plt.show()


In [ ]:
# Cell 5 — Per-solvent breakdown plot

by_solvent = report.get("by_solvent", {})
if len(by_solvent) > 2:
    names = list(by_solvent.keys())
    maes = [by_solvent[n]["mae"] for n in names]
    counts = [by_solvent[n]["n"] for n in names]

    # Sort by count
    order = np.argsort(counts)[::-1]
    names = [names[i] for i in order]
    maes = [maes[i] for i in order]
    counts = [counts[i] for i in order]

    fig, ax1 = plt.subplots(figsize=(10, 5))

    x = np.arange(len(names))
    bars = ax1.bar(x, maes, color="steelblue", alpha=0.8)
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
    ax1.set_ylabel("MAE (ln x₂)")
    ax1.set_title("MAE by solvent")

    # Overlay count
    ax2 = ax1.twinx()
    ax2.plot(x, counts, "ro-", ms=5, alpha=0.7, label="n records")
    ax2.set_ylabel("N records")
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(str(NOTEBOOK_FIG_DIR / "per_solvent_mae.png"), dpi=150)
    plt.show()


In [ ]:
# Cell 6 — MC-Dropout uncertainty

mc = MCDropoutPredictor(model, n_samples=30)

# Pick representative systems
test_systems = [
    ("CC(=O)Nc1ccc(O)cc1", "CCO", 298.15),       # paracetamol / ethanol
    ("CC(=O)Nc1ccc(O)cc1", "O", 298.15),           # paracetamol / water
    ("c1ccc2ccccc2c1", "c1ccccc1", 298.15),         # naphthalene / benzene
    ("OC(=O)c1ccccc1", "O", 298.15),                # benzoic acid / water
    ("CC(=O)Oc1ccccc1C(=O)O", "CCO", 298.15),      # aspirin / ethanol
    ("CCCCCCCC", "O", 298.15),                       # octane / water
]

mc_results = mc.predict_batch(test_systems)

display_cols = [
    "solute", "solvent",
    "ln_x2_mean", "ln_x2_std", "ln_x2_q05", "ln_x2_q95",
    "T_m_mean", "T_m_std",
]
print("MC-Dropout uncertainty estimates:")
print(mc_results[display_cols].to_string(
    index=False, float_format="{:.3f}".format
))

In [ ]:
# Cell 7 — Uncertainty visualization

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Error bars
ax = axes[0]
for i, (_, row) in enumerate(mc_results.iterrows()):
    label = f"{row['solute'][:15]}/{row['solvent'][:8]}"
    mean = row["ln_x2_mean"]
    lo = row["ln_x2_q05"]
    hi = row["ln_x2_q95"]
    ax.errorbar(i, mean, yerr=[[mean - lo], [hi - mean]],
                fmt="o", capsize=4, capthick=1.5, ms=6)
ax.set_xticks(range(len(mc_results)))
ax.set_xticklabels(
    [f"{r['solute'][:12]}\n{r['solvent'][:8]}"
     for _, r in mc_results.iterrows()],
    fontsize=7, rotation=45, ha="right",
)
ax.set_ylabel("ln(x₂)")
ax.set_title("Predictions with 90% CI (MC-Dropout)")

# Uncertainty vs prediction magnitude
ax = axes[1]
ax.scatter(
    mc_results["ln_x2_mean"].abs(),
    mc_results["ln_x2_std"],
    s=80, c="coral", edgecolors="black",
)
ax.set_xlabel("|ln(x₂)| (magnitude)")
ax.set_ylabel("σ (uncertainty)")
ax.set_title("Uncertainty vs solubility magnitude")

plt.tight_layout()
plt.savefig(str(NOTEBOOK_FIG_DIR / "uncertainty_plots.png"), dpi=150)
plt.show()


In [ ]:
# Cell 8 — Calibration check (if we have ground truth for test systems)

# For full calibration, run MC-Dropout on entire test set
# Here we demonstrate the API with the small test batch

test_true = []
for sol, slv, T in test_systems:
    match = test_df[
        (test_df["solute_smiles"] == sol) &
        (test_df["solvent_smiles"] == slv) &
        (test_df["has_solubility"] == True)
    ]
    if len(match) > 0:
        test_true.append(match.iloc[0]["ln_x2"])
    else:
        test_true.append(None)

# Filter to systems with known values
valid_preds = []
valid_true = []
for pred_row, true_val in zip(mc_results.to_dict("records"), test_true):
    if true_val is not None:
        valid_preds.append(pred_row)
        valid_true.append(true_val)

if len(valid_preds) >= 3:
    cal = calibration_report(valid_preds, valid_true)
    print("Calibration report:")
    for k, v in cal.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    print("Not enough matched systems for calibration check")

In [ ]:
# Cell 9 — Applicability Domain

ad = ApplicabilityDomain(model, train_loader)

# Test known in-domain systems
print("=== In-domain systems ===")
for sol, slv, T in test_systems[:3]:
    print(ad.report(sol, slv, T))
    print()

# Test potentially OOD systems
print("=== Potentially OOD systems ===")
ood_systems = [
    # Novel heterocycle not in training
    ("c1cnc2[nH]cnc2c1", "O", 298.15),
    # Fluorinated compound
    ("FC(F)(F)c1ccc(O)cc1", "CCCCCC", 298.15),
    # Organometallic-like
    ("c1ccc([Se]c2ccccc2)cc1", "CCO", 298.15),
]

for sol, slv, T in ood_systems:
    try:
        print(ad.report(sol, slv, T))
    except ValueError as e:
        print(f"Cannot process {sol}: {e}")
    print()

In [ ]:
# Cell 10 — AD confidence vs actual error

# Score all test predictions for AD confidence
print("Computing AD scores for test set...")
test_sol_df = test_df[test_df["has_solubility"]].reset_index(drop=True)
n_check = min(200, len(test_sol_df))  # limit for speed

ad_confs = []
ad_errors = []
for i in range(n_check):
    row = test_sol_df.iloc[i]
    try:
        score = ad.score(
            row["solute_smiles"], row["solvent_smiles"],
            row["temperature"],
        )
        r = predict_solubility(
            model, row["solute_smiles"], row["solvent_smiles"],
            row["temperature"],
        )
        error = abs(r["ln_x2"] - row["ln_x2"])
        ad_confs.append(score["confidence"])
        ad_errors.append(error)
    except Exception:
        continue

if len(ad_confs) > 10:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(ad_confs, ad_errors, s=15, alpha=0.5, c="steelblue")
    ax.set_xlabel("AD confidence")
    ax.set_ylabel("|Error| in ln(x₂)")
    ax.set_title("Applicability Domain: confidence vs actual error")

    # Trend line
    from numpy.polynomial import polynomial as P
    coef = P.polyfit(ad_confs, ad_errors, 1)
    x_fit = np.linspace(0, 1, 50)
    ax.plot(x_fit, P.polyval(x_fit, coef), "r--", lw=1.5,
            label=f"trend (slope={coef[1]:.2f})")
    ax.legend()

    plt.tight_layout()
    plt.savefig(str(NOTEBOOK_FIG_DIR / "ad_confidence_vs_error.png"), dpi=150)
    plt.show()

    # Correlation
    corr = np.corrcoef(ad_confs, ad_errors)[0, 1]
    print(f"Correlation(confidence, |error|): {corr:.3f}")
    print(f"  (negative = higher confidence → lower error = good)")
